# SC-12-Move-Sui - Move sur Sui

**Navigation** : [Index](../README.md) | [<< Formal Verification](../03-Foundry-Testing/SC-11-Formal-Verification.ipynb) | [Solana >>](SC-13-Solana-Anchor.ipynb)

---

## Objectifs d'apprentissage

1. Decouvrir le langage **Move** (Meta/Diem)
2. Comprendre le **modele objet** de Sui
3. Creer un module Move simple
4. Utiliser les **abilities** (key, store, copy, drop)

### Prerequis

- [SC-1](../01-Solidity-Foundation/SC-1-Solidity-Basics.ipynb) a [SC-4](../01-Solidity-Foundation/SC-4-Errors-Events.ipynb) completes (concepts)
- Notions de programmation orientee ressources
- Sui CLI installe (voir instructions dans le notebook)

### Duree estimee : 50 minutes

---

## 1. Introduction a Move

Move est un langage concu pour les smart contracts avec une semantique de ressources.

In [1]:
# Comparaison Move vs Solidity
print("""
MOVE vs SOLIDITY

| Caracteristique    | Solidity         | Move                |
|-------------------|------------------|---------------------|
| Paradigme         | Comptes          | Objets              |
| Ressources        | Non-lineaires    | Lineaires (assets)  |
| Ownership         | Implicit         | Explicit (abilities)|
| Generique         | Non              | Oui                 |
| Bytecode          | EVM              | Move VM             |

ABILITIES (Capacites):
- key    : Peut etre un identifiant global
- store  : Peut etre stocke dans un objet
- copy   : Peut etre copie
- drop   : Peut etre detruit implicitement
""")


MOVE vs SOLIDITY

| Caracteristique    | Solidity         | Move                |
|-------------------|------------------|---------------------|
| Paradigme         | Comptes          | Objets              |
| Ressources        | Non-lineaires    | Lineaires (assets)  |
| Ownership         | Implicit         | Explicit (abilities)|
| Generique         | Non              | Oui                 |
| Bytecode          | EVM              | Move VM             |

ABILITIES (Capacites):
- key    : Peut etre un identifiant global
- store  : Peut etre stocke dans un objet
- copy   : Peut etre copie
- drop   : Peut etre detruit implicitement



---

## 2. Module Move Simple

Exemple d'un token simple sur Sui.

In [2]:
# Module Move: Coin natif
MOVE_COIN = '''
/// Module: my_coin.move
module my_package::my_coin {
    use sui::coin::{Self, Coin};
    use sui::transfer;
    use sui::object::{Self, UID};
    use sui::tx_context::{Self, TxContext};

    /// Erreur: montant insuffisant
    const EInsufficientBalance: u64 = 0;

    /// Structure du token (capability pour mint)
    public struct AdminCap has key {
        id: UID,
    }

    /// One-Time Witness pour le coin
    public struct MY_COIN has drop {}

    /// Initialisation du module
    fun init(witness: MY_COIN, ctx: &mut TxContext) {
        // Creer la capability admin
        transfer::transfer(
            AdminCap { id: object::new(ctx) },
            tx_context::sender(ctx)
        );

        // Enregistrer le coin
        coin::create_currency(witness, 9, b"MYC", b"My Coin", b"", ctx);
    }

    /// Minter de nouveaux tokens
    public fun mint(
        _: &AdminCap,
        amount: u64,
        recipient: address,
        ctx: &mut TxContext
    ) {
        let coin = coin::mint(amount, ctx);
        transfer::public_transfer(coin, recipient);
    }
}
'''

print("Module Move - Coin:")
print(MOVE_COIN)

Module Move - Coin:

/// Module: my_coin.move
module my_package::my_coin {
    use sui::coin::{Self, Coin};
    use sui::transfer;
    use sui::object::{Self, UID};
    use sui::tx_context::{Self, TxContext};

    /// Erreur: montant insuffisant
    const EInsufficientBalance: u64 = 0;

    /// Structure du token (capability pour mint)
    public struct AdminCap has key {
        id: UID,
    }

    /// One-Time Witness pour le coin
    public struct MY_COIN has drop {}

    /// Initialisation du module
    fun init(witness: MY_COIN, ctx: &mut TxContext) {
        // Creer la capability admin
        transfer::transfer(
            AdminCap { id: object::new(ctx) },
            tx_context::sender(ctx)
        );

        // Enregistrer le coin
        coin::create_currency(witness, 9, b"MYC", b"My Coin", b"", ctx);
    }

    /// Minter de nouveaux tokens
    public fun mint(
        _: &AdminCap,
        amount: u64,
        recipient: address,
        ctx: &mut TxContext
    ) {
    

---

## 3. Modele Objet Sui

Chaque objet a un proprietaire unique.

In [3]:
# NFT simple sur Sui
MOVE_NFT = '''
module my_package::my_nft {
    use sui::object::{Self, UID};
    use sui::transfer;
    use sui::tx_context::{Self, TxContext};
    use sui::url::Url;

    /// Structure NFT
    public struct NFT has key, store {
        id: UID,
        name: String,
        description: String,
        url: Url,
    }

    /// Admin capability pour minter
    public struct AdminCap has key { id: UID }

    /// Initialisation
    fun init(ctx: &mut TxContext) {
        transfer::transfer(
            AdminCap { id: object::new(ctx) },
            tx_context::sender(ctx)
        );
    }

    /// Minter un nouveau NFT
    public fun mint(
        _: &AdminCap,
        name: String,
        description: String,
        url: Url,
        recipient: address,
        ctx: &mut TxContext
    ) {
        let nft = NFT {
            id: object::new(ctx),
            name,
            description,
            url,
        };
        transfer::public_transfer(nft, recipient);
    }

    /// Transferer un NFT
    public fun transfer(nft: NFT, recipient: address) {
        transfer::public_transfer(nft, recipient);
    }

    /// Burn un NFT
    public fun burn(nft: NFT) {
        let NFT { id, name: _, description: _, url: _ } = nft;
        object::delete(id);
    }
}
'''

print("Module Move - NFT:")
print(MOVE_NFT)

Module Move - NFT:

module my_package::my_nft {
    use sui::object::{Self, UID};
    use sui::transfer;
    use sui::tx_context::{Self, TxContext};
    use sui::url::Url;

    /// Structure NFT
    public struct NFT has key, store {
        id: UID,
        name: String,
        description: String,
        url: Url,
    }

    /// Admin capability pour minter
    public struct AdminCap has key { id: UID }

    /// Initialisation
    fun init(ctx: &mut TxContext) {
        transfer::transfer(
            AdminCap { id: object::new(ctx) },
            tx_context::sender(ctx)
        );
    }

    /// Minter un nouveau NFT
    public fun mint(
        _: &AdminCap,
        name: String,
        description: String,
        url: Url,
        recipient: address,
        ctx: &mut TxContext
    ) {
        let nft = NFT {
            id: object::new(ctx),
            name,
            description,
            url,
        };
        transfer::public_transfer(nft, recipient);
    }

    ///

---

## 4. Commandes Sui CLI

Compilation et deploiement.

In [4]:
# Commandes Sui CLI
print("""
INSTALLATION:
cargo install --locked --git https://github.com/MystenLabs/sui.git --branch mainnet sui

CONFIGURATION:
sui client active-address     # Voir l'adresse active
sui client active-env        # Voir l'environnement
sui client switch --env devnet  # Changer d'environnement

PROJET:
sui move new my_package    # Creer un nouveau projet
cd my_package
mkdir -p sources tests

COMPILATION:
sui move build            # Compiler le module

TESTS:
sui move test             # Executer les tests unitaires

DEPLOIEMENT:
sui client publish --gas-budget 100000000

APPELS:
sui client call --package 0x... --module my_coin \
    --function mint --args 1000 0x... --gas-budget 10000000
""")


INSTALLATION:
cargo install --locked --git https://github.com/MystenLabs/sui.git --branch mainnet sui

CONFIGURATION:
sui client active-address     # Voir l'adresse active
sui client active-env        # Voir l'environnement
sui client switch --env devnet  # Changer d'environnement

PROJET:
sui move new my_package    # Creer un nouveau projet
cd my_package
mkdir -p sources tests

COMPILATION:
sui move build            # Compiler le module

TESTS:
sui move test             # Executer les tests unitaires

DEPLOIEMENT:
sui client publish --gas-budget 100000000

APPELS:
sui client call --package 0x... --module my_coin     --function mint --args 1000 0x... --gas-budget 10000000



---

## 5. Exercices

In [5]:
# Exercice: Escrow simple
EXERCISE_ESCROW = '''
module my_package::escrow {
    use sui::object::{Self, UID};
    use sui::transfer;
    use sui::coin::Coin;
    use sui::sui::SUI;
    use sui::tx_context::{Self, TxContext};

    /// Erreurs
    const ENotSeller: u64 = 0;
    const ENotCompleted: u64 = 1;

    /// Escrow object
    public struct Escrow has key {
        id: UID,
        seller: address,
        buyer: address,
        amount: u64,
        completed: bool,
    }

    /// Creer un escrow
    public fun create(
        seller: address,
        buyer: address,
        payment: Coin<SUI>,
        ctx: &mut TxContext
    ): Escrow {
        Escrow {
            id: object::new(ctx),
            seller,
            buyer,
            amount: coin::value(&payment),
            completed: false,
        }
    }

    /// Completer l'escrow
    public fun complete(
        escrow: &mut Escrow,
        sender: address,
        payment: Coin<SUI>,
        ctx: &mut TxContext
    ) {
        assert!(sender == escrow.buyer, ENotSeller);
        escrow.completed = true;
        transfer::public_transfer(payment, escrow.seller);
    }
}
'''
print("Exercice Escrow:")
print(EXERCISE_ESCROW)

Exercice Escrow:

module my_package::escrow {
    use sui::object::{Self, UID};
    use sui::transfer;
    use sui::coin::Coin;
    use sui::sui::SUI;
    use sui::tx_context::{Self, TxContext};

    /// Erreurs
    const ENotSeller: u64 = 0;
    const ENotCompleted: u64 = 1;

    /// Escrow object
    public struct Escrow has key {
        id: UID,
        seller: address,
        buyer: address,
        amount: u64,
        completed: bool,
    }

    /// Creer un escrow
    public fun create(
        seller: address,
        buyer: address,
        payment: Coin<SUI>,
        ctx: &mut TxContext
    ): Escrow {
        Escrow {
            id: object::new(ctx),
            seller,
            buyer,
            amount: coin::value(&payment),
            completed: false,
        }
    }

    /// Completer l'escrow
    public fun complete(
        escrow: &mut Escrow,
        sender: address,
        payment: Coin<SUI>,
        ctx: &mut TxContext
    ) {
        assert!(sender == es

---

## 6. Resume

| Concept | Description |
|---------|-------------|
| `module` | Unite de code Move |
| `struct` | Structure de donnees avec abilities |
| `key` | Peut etre un objet global |
| `store` | Peut etre stocke dans un autre objet |
| `UID` | Identifiant unique d'objet |
| `TxContext` | Contexte de transaction |

---

**Notebook suivant** : [SC-13-Solana-Anchor](SC-13-Solana-Anchor.ipynb)